# TrainLM: native TorchTPU pretraining from scratch

Open in Colab on a single-device Cloud TPU v5e runtime. This initializes a GPT-2 architecture with random weights and trains it on a pinned Hugging Face text dataset (TinyStories by default). Set the dataset ID, immutable revision, split names, and text column in the configuration cell to use your own corpus. It trains a byte-level BPE tokenizer using only the selected training split, appends EOS boundaries, writes validated uint32 packed shards with document offsets, and reads fixed-shape examples through TrainLM's PackedBinDataset. The notebook builds the GPT-2 config locally from visible layer, width, head, sequence-length, and vocabulary settings; no pretrained model config, weights, or tokenizer are downloaded. Defaults run 1,000 optimizer steps with validation and periodic checkpoints. This is a small-scale training recipe, not a production-scale model or performance claim.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path(tempfile.mkdtemp(prefix="trainlm-native-scratch-source-")) / "TrainLM"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch",
    "main",
    "https://github.com/Dhiraj309/TrainLM.git", str(repo_root),
], check=True)
source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()
print(f"source_revision={source_revision}")

## Configure and check the runtime

The package check runs before dataset files are downloaded or tokenizer training starts. Model dimensions, vocabulary, pinned dataset commit, splits, and text column are set in the next cell. Use distinct, non-overlapping training and validation data; a mutable dataset branch is rejected. Reduce the example counts and step count for a quick smoke; increase them for a longer pretraining run. Save and validation cadence follow the requested step count.

In [ ]:
import json
import runpy

TRAINING_STEPS = 1000
TRAIN_EXAMPLES = 10000
DATASET_ID = "roneneldan/TinyStories"
DATASET_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"
TRAIN_SPLIT = "train"
VALIDATION_SPLIT = "validation"
TEXT_FIELD = "text"
TOKENIZER_VOCAB_SIZE = 16000
MODEL_EMBEDDING_SIZE = 768
MODEL_LAYERS = 6
MODEL_ATTENTION_HEADS = 12
VALIDATION_EXAMPLES = 1000
SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 8
MAX_EVAL_BATCHES = 20
SAVE_EVERY = 100
session_dir = Path(tempfile.mkdtemp(prefix="trainlm-native-scratch-run-"))
report_path = session_dir / "trainlm_native_scratch_report.json"
inspect_profile = runpy.run_path(str(repo_root / "examples/torchtpu_t1_install_preflight.py"))["inspect_profile"]
assess_profile = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_trainer_preflight.py"))["assess_trainer_profile"]
preflight = assess_profile(inspect_profile(repo_root), require_datasets=True)
print(json.dumps(preflight, indent=2))
if not preflight["proceed"]:
    report = {"schema_version": 1, "scope": "single_device_native_scratch_pretraining",
              "status": "blocked", "source_revision": source_revision,
              "preflight": preflight, "checks": {"train": "not_run",
              "export": "not_run", "compile": "not_run",
              "resume": "not_run", "exact_weight_match": "not_run"}}
    report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(f"shareable_report={report_path}")
    raise RuntimeError("Package preflight blocked pretraining; download the report for details.")

## Train and export

This child process uses TrainLM's explicit experimental `native_tpu` path. It tokenizes pinned text streams into validated uint32 packed shards, trains from random initialization, evaluates on the pinned validation split, saves periodic checkpoints, and exports ordinary Hugging Face model/tokenizer files. The process is time-bounded and raw logs stay local.

In [ ]:
import json
import runpy

run_process = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_probe_process.py"))["run_probe_process"]
command = [sys.executable, str(repo_root / "examples/torchtpu_native_trainer_smoke.py"),
           "--output-dir", str(session_dir / "artifacts"), "--from-scratch",
           "--max-steps", str(TRAINING_STEPS),
           "--train-examples", str(TRAIN_EXAMPLES),
           "--tokenizer-vocab-size", str(TOKENIZER_VOCAB_SIZE),
           "--model-embedding-size", str(MODEL_EMBEDDING_SIZE),
           "--model-layers", str(MODEL_LAYERS),
           "--model-attention-heads", str(MODEL_ATTENTION_HEADS),
           "--dataset-id", DATASET_ID,
           "--dataset-revision", DATASET_REVISION,
           "--train-split", TRAIN_SPLIT,
           "--validation-split", VALIDATION_SPLIT,
           "--text-field", TEXT_FIELD,
           "--eval-examples", str(VALIDATION_EXAMPLES),
           "--sequence-length", str(SEQUENCE_LENGTH),
           "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
           "--max-eval-batches", str(MAX_EVAL_BATCHES),
           "--save-steps", str(SAVE_EVERY)]
result = run_process(command, cwd=repo_root, timeout=3600)
(session_dir / "stdout.log").write_text(result["stdout"])
(session_dir / "stderr.log").write_text(result["stderr"])
build_report = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_trainer_evidence.py"))["build_trainer_report"]
report = build_report(result, source_revision=source_revision,
                      compile_requested=False, resume_requested=False)
report["scope"] = "single_device_native_scratch_pretraining"
report["preflight"] = preflight
report["limitations"] = ["experimental_single_device", "does_not_certify_performance"]
if (report["status"] == "passed" and
        (report["model"] is None or report["model"].get("initialization") != "random_from_config"
         or report["dataset"] is None)):
    report["status"] = "failed"
    report["issues"] = sorted(set(report["issues"] + ["scratch_provenance_missing"]))
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
print(json.dumps(report, indent=2))
print(f"shareable_report={report_path}")
print(f"local_artifacts_and_logs={session_dir}")
if report["status"] != "passed":
    raise RuntimeError("Scratch pretraining failed; inspect local logs before retrying.")

## Download the report

Run this after the training cell, including when it blocks or fails. The report includes source, model, dataset, device, package, and metric evidence; raw logs remain in the Colab runtime.

In [ ]:
from google.colab import files
files.download(str(report_path))